# MoStegoLLM — Interactive Demo

This notebook demonstrates how MoStegoLLM hides binary data inside natural English prose using arithmetic coding over LLM token probability distributions.

In [ ]:
from mostegollm import StegoCodec

codec = StegoCodec(device="auto")
print("Model loaded!")

## Basic Round-Trip
Encode a secret message into cover text, then decode it back.

In [ ]:
secret = "attack at dawn"
cover = codec.encode_str(secret)
print(f"Secret:  {secret}")
print(f"Cover:   {cover}")
print(f"Decoded: {codec.decode_str(cover)}")

## Encryption
Add AES-256-GCM encryption for confidentiality on top of steganographic hiding.

In [ ]:
secure = StegoCodec(device="auto", password="my-key")
# Share the already-loaded model
secure._model, secure._tokenizer, secure._device = codec._ensure_model()

cover = secure.encode(b"classified")
print(f"Encrypted cover: {cover[:100]}...")
print(f"Recovered: {secure.decode(cover)}")

## Capacity: Payload Size vs. Cover Text Length
How many tokens does it take to encode different payload sizes?

In [ ]:
import os

sizes = [5, 10, 25, 50, 100, 200]
results = []
for size in sizes:
    data = os.urandom(size)
    stats = codec.encode_with_stats(data)
    results.append((size, stats.total_tokens, stats.bits_per_token))
    print(f"{size:>4} bytes -> {stats.total_tokens:>4} tokens  ({stats.bits_per_token:.1f} bits/token)")

## How It Works: Token Probability Distributions
At each step, the LLM produces a probability distribution over the next token. The encoder selects tokens based on the secret data bits. Here's what one step looks like:

In [ ]:
import torch
from mostegollm.encoder import _get_token_distribution

model, tokenizer, device = codec._ensure_model()
prompt_ids = tokenizer.encode("The weather today is", return_tensors="pt").to(device)

tok_ids, cum_probs, _ = _get_token_distribution(model, prompt_ids, device, top_k=20)

# Show top-20 tokens and their probabilities
tokens = [tokenizer.decode([tid]) for tid in tok_ids]
widths = [(cum_probs[i+1] - cum_probs[i]) / (2**32) * 100 for i in range(len(tok_ids))]

print(f"{'Token':<20} {'Probability':>12}")
print("-" * 34)
for tok, w in zip(tokens, widths):
    bar = "#" * int(w * 2)
    print(f"{tok!r:<20} {w:>10.1f}%  {bar}")

## Chunked Encoding
For large payloads, split into independently encoded chunks:

In [ ]:
large = b"A" * 200
covers = codec.encode(large, chunk_size=50)
print(f"{len(large)} bytes -> {len(covers)} chunks")
for i, c in enumerate(covers):
    print(f"  Chunk {i+1}: {c[:60]}...")

recovered = codec.decode(covers)
assert recovered == large
print(f"\nRecovered {len(recovered)} bytes successfully!")